# Exploring & Cleaning Data with Pandas

**Dataset:** `Sample-Superstore2019.csv` — 9,994 orders from a fictional retail superstore (2016–2019), with columns covering customers, products, shipping, and sales/profit figures.

**Goal of this lecture:** build a repeatable workflow for taking a raw CSV from "just downloaded" to "ready for analysis" — inspecting it, understanding its shape and types, finding and fixing problems, and confirming it's trustworthy before we build anything on top of it.

## Agenda
0. Setup
1. Filtering, Selecting & Boolean Indexing
2. Getting Oriented — Reading & First Look
3. Descriptive Exploration
4. Data Types & Conversions
5. Missing Data
6. Duplicate Detection
7. Inconsistent / Dirty Values
8. Outliers & Sanity Checks
9. Wrap-Up — A Repeatable Cleaning Checklist

Each topic follows the same pattern: **explanation → demo → exercise**. Exercise cells are left blank (with guiding comments) for you to fill while you are studying at home.

## 0. Setup

Before exploring anything we need our tools and our data in memory.

- `pandas` is the core library for tabular data — it gives us the `DataFrame`, the object we'll spend the whole lecture inside.
- `numpy` occasionally helps with numeric operations (e.g. representing missing values, vectorized math).
- `matplotlib` will let us draw quick sanity-check plots (histograms, boxplots) later on.
> **Important Note:**
We load the CSV with `pd.read_csv()`. This one file happens to load fine with the default UTF-8 encoding — but keep in mind that real-world CSVs (especially exports from Excel) often need `encoding="latin1"` or `encoding="cp1252"`. If `read_csv` ever throws a `UnicodeDecodeError`, that's your cue to try a different encoding.

In [ ]:
# Core libraries for this lecture
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow
# Read the CSV into a DataFrame.
# If this ever raises UnicodeDecodeError on another file, try encoding="latin1".
df = pd.read_csv("Sample-Superstore2019.csv")

# Quick confirmation that the file loaded and how big it is (rows, columns)
df.shape

> I think it will be a best practice to copy the original dataframe into a new one and start working on the new one without a touch to the original one.

In [ ]:
df_raw = df.copy()
df_raw.shape

## 1. Filtering, Selecting & Boolean Indexing

Cleaning inevitably raises questions like "show me just the rows where X is true" — this is where `.loc`, `.iloc`, and boolean indexing come in.

- `.loc[row_selector, column_selector]` selects by **label** (column names, index labels, or a boolean condition).
- `.iloc[row_positions, column_positions]` selects by **integer position**, regardless of labels.
- A boolean condition like `df["Profit"] < 0` produces a `True`/`False` Series the same length as `df`; passing it into `df[...]` or `df.loc[...]` filters to just the matching rows.
- Conditions can be combined with `&` (and), `|` (or) — each condition **must** be wrapped in parentheses because of Python operator precedence.

These techniques are what let us act on everything we found while cleaning — e.g. actually pulling up the loss-making orders we noticed in Section 7.

In [ ]:
# We learn the difference between the .loc and .iloc very well as it is important
# .loc() uses the labels of both the row and column
# where .iloc() uses the position expressed in integer value for both the row and the column

# Normally, we are going to use the .loc() method to select the rows and columns using the labels

In [ ]:
# Boolean indexing: orders sold at a loss
loss_orders = df[df["Profit"] < 0]
print("Loss-making orders:", len(loss_orders))
loss_orders[["Order ID", "Product Name", "Sales", "Profit"]].head()

In [ ]:
# Combine multiple conditions with & (remember the parentheses around each condition!)
same_day_west = df[(df["Ship Mode"] == "Same Day") & (df["Region"] == "West")]
same_day_west[["Order ID", "Region", "Ship Mode", "Sales"]].head()

In [ ]:
# .loc for label-based selection: rows where Profit < 0, only a few relevant columns
df.loc[df["Profit"] < 0, ["Order ID", "Category", "Sub-Category", "Profit"]].head()

# .iloc for pure position-based selection: first 5 rows, first 3 columns
df.iloc[:5, :3]

### 🧪 Exercise 1 — Filtering, Selecting & Boolean Indexing

1. Filter `df` to only orders in the `Technology` category with `Sales` greater than 1000.
2. Using `.loc`, select the `Order ID`, `Customer Name`, and `Profit` columns for all orders where `Discount` is greater than 0.5.
3. Using `.iloc`, select the last 5 rows and the last 4 columns of `df`.
4. Combine three conditions: orders from the `Consumer` segment, in the `Central` region, shipped `Standard Class` — how many orders match all three?

In [ ]:
# TODO 1: Technology orders with Sales > 1000


# TODO 2: .loc selection -- Order ID, Customer Name, Profit where Discount > 0.5


# TODO 3: .iloc -- last 5 rows, last 4 columns


# TODO 4: combine three conditions with &


## 2. Getting Oriented — Reading & First Look

The very first thing to do with any new dataset is **look at it before you touch it**. We want to answer:

- How many rows and columns do we have? (`.shape`)
- What do the first/last few rows actually look like? (`.head()`, `.tail()`)
- What are the column names, and what type does pandas think each one is? (`.columns`, `.dtypes`, `.info()`)

`.info()` is especially useful because it combines row count, column names, non-null counts, and dtypes in one view — it's usually the first command to run on any new DataFrame.

Watch for the `Unnamed: 0` column in the output below. That's a classic artifact of a CSV that was exported *with* its row index included — it's not real data, and cleaning it up (or telling `read_csv` to use it as the index) is itself a small first cleaning step.

> Dropping the columns we do not need should be done using the .drop(subset=[]) method, as I think this will makes more sense in this manner.

In [ ]:
# In real-life project i'd drop the postal code column but I will leave it as it is to practice
# detecting the missing data and filling it.
# this is the cleanest pattern for dropping columns from the DataFrame using .drop() function.
df_raw = df_raw.drop(columns=["Unnamed: 0", "Row ID"])
df_raw.info(memory_usage='deep')

> I think adding the step of case & extra spaces check before converting the data types is mandatory!

In [ ]:
string_cols = df_raw.select_dtypes('str').columns.tolist()

for col in string_cols:
    # Striping the extra spaces and converting the case into the title case,
    # the first letter is capital and the remaining letters are small
    df_raw[col] = df_raw[col].str.strip().str.title()

In [ ]:
# Shape: (number of rows, number of columns)
print("Shape:", df_raw.shape)

# First and last rows — sanity check that data looks like what we expect
df_raw.head()

In [ ]:
# Exploring the tail of the dataset is mandatory as we should detect if there is any metadata
# exists in the tail of the data...
df_raw.tail()

In [ ]:
# Column names
print(df_raw.columns.tolist())

# .info() gives dtypes + non-null counts + memory usage all at once
df_raw.info()

### 🧪 Exercise 2 — Getting Oriented

1. Print only the column names of `df` as a Python list.
2. Display the first **7** rows instead of the default 5.
3. Using `.info()`, answer: which column(s), if any, have fewer non-null values than the total row count?
4. `df` was loaded with an extra `Unnamed: 0` column. Look at `pd.read_csv`'s documentation (or `help(pd.read_csv)`) and find the parameter that would let you treat that column as the row index at load time instead.

In [ ]:
# TODO 1: print the column names as a list


# TODO 2: display the first 7 rows


# TODO 3: inspect df.info() output and note the column(s) with missing values (write your answer as a comment)


# TODO 4: re-read the CSV using a read_csv parameter that avoids the Unnamed: 0 column


## 3. Descriptive Exploration

Once we know the shape of the data, the next question is: **what values actually live in each column?**

- `.describe()` on numeric columns gives count, mean, std, min, quartiles, and max in one call — a fast way to spot impossible values (e.g. a negative quantity) or extreme outliers hiding in the min/max.
- `.describe(include="object")` (or `include="all"`) does the equivalent for text/categorical columns: count, number of unique values, the most frequent value, and its frequency.
- `.value_counts()` on a single categorical column shows exactly how many rows fall into each category — useful for spotting typos ("Consumer" vs "consumer") or a category that's barely represented.

The goal here isn't to fix anything yet — it's to build intuition about what "normal" looks like in this dataset, so that later, unusual values jump out.

In [ ]:
# Leveraging the summary statistics for the numeric columns is not about all the numeric columns
# but about the numeric columns that make sense to be statistically described
# for example, describing the postal code statistically does not make sense at alll
# So, we need to specifiy the DataFrame subset to be statistically described
df_raw[["Sales", "Quantity", "Discount", "Profit"]].describe()

In [ ]:
# Summary stats for text/categorical columns
df_raw.describe(include="str")

In [ ]:
df_loss = df_raw[(df_raw["Profit"]<0) & (df_raw["Discount"] == 0.8)]
df_loss

In [ ]:
# This should be identical to the previous one except using the filtering mask
mask = ((df_raw["Profit"] < 0) & (df_raw["Discount"] == df_raw["Discount"].max()))

df_loss_mask = df_raw[mask]
df_loss_mask

In [ ]:
# How many orders per Region / Segment / Ship Mode?
# Now, we already cleaned the extra space and the inconsistent wording
print(df_raw["Region"].value_counts())
print()
print(df_raw["Segment"].value_counts())
print()
print(df_raw["Ship Mode"].value_counts())

In [ ]:
df_raw["Region"].value_counts().plot(kind='bar')
plt.title("Order by Region")
plt.xlabel("Region")
plt.ylabel("Number of Orders")
plt.show()

### 🧪 Exercise 3 — Descriptive Exploration

1. Run `.describe()` on just the `Sales` and `Profit` columns (hint: select the columns first, then call `.describe()`).
2. Use `.value_counts()` on `Category` — which category has the most orders?
3. Use `.value_counts(normalize=True)` on `Ship Mode` to see the *percentage* of orders per shipping mode instead of raw counts.
4. Looking at the `.describe()` output for `Discount`, does the minimum/maximum look like a plausible discount range? Write your answer as a comment.

In [ ]:
# TODO 1: describe() on Sales and Profit only
print(df_raw[["Sales", "Profit"]].describe())


# TODO 2: value_counts() on Category
print(df_raw["Category"].value_counts())


# TODO 3: value_counts(normalize=True) on Ship Mode
print(df_raw["Ship Mode"].value_counts(normalize=True)*100)


# TODO 4: comment on whether Discount's min/max look plausible
print(f"Minimum Discount: {df_raw["Discount"].min()} - Maximum Discount: {df_raw["Discount"].max()}")


In [ ]:
df_raw["Category"].value_counts().plot(kind='bar')
plt.title("Orders By Category")
plt.xlabel("Product Category")
plt.ylabel("Number of Orders")
plt.show()

## 4. Data Types & Conversions

`.info()` already told us the dtype of every column — but dtype is about more than trivia. The *type* pandas assigns determines what operations are even possible:

- `Order Date` and `Ship Date` are loaded as plain `object` (string) columns, not dates — so right now we can't subtract one from the other, sort chronologically, or pull out "just the month." Converting them with `pd.to_datetime()` unlocks all of that.
- Columns with a small, fixed set of repeated text values (like `Segment`, `Region`, `Ship Mode`) are good candidates for the `category` dtype — it uses far less memory and speeds up groupby operations on large datasets.

A good habit: after loading any new CSV, check `.dtypes` and ask "is this the type I actually need, or just the type pandas guessed?" 

In [ ]:
df_raw.info(memory_usage='deep')

In [ ]:
# After checking the data type of each column in the DataFrame we can mention the following
# We need to convert both 'Order Date' and 'Ship Date' into datetime data type
# Here, we can convert each column separately or using the .apply() function and pass the function
# name to it pd.to_datetime - without paranthesis ...
df_raw[["Order Date", "Ship Date"]] = df_raw[["Order Date", "Ship Date"]].apply(pd.to_datetime)
df_raw.info()

In [ ]:
# The result of this cell should be identical with the previous one with a single difference between both of them
# here we are going to convert each column separately
# df_raw["Order Date"] = pd.to_datetime(df_raw["Order Date"])
# df_raw["Ship Date"] = pd.to_datetime(df_raw["Ship Date"])
# df_raw.info()

In [ ]:
# Now, we are going to convert the string data types with low unique values - low cardinality - into category data type
str_cols = df_raw.select_dtypes('str').columns.tolist()

for col in str_cols:
    if len(df_raw[col].unique()) <= 50:
        df_raw[col] = df_raw[col].astype("category")

df_raw.info()

In [ ]:
print(df_raw.memory_usage(deep=True))
print(df_raw.info(memory_usage="deep"))

### 🧪 Exercise 4 — Data Types & Conversions

1. Confirm the conversion worked by checking the dtype of `Order Date` (should be `datetime64[ns]`).
2. Create a new column `Order Weekday` containing the name of the weekday each order was placed on (hint: `.dt.day_name()`).
3. What is the *average* `Shipping Days` across all orders?
4. Convert `Sub-Category` to the `category` dtype as well, the same way we did for `Segment`.

In [ ]:
# TODO 1: confirm Order Date's dtype


# TODO 2: create an "Order Weekday" column using .dt.day_name()


# TODO 3: compute the average Shipping Days


# TODO 4: convert Sub-Category to category dtype

# TODO 5: Check the numeric data types and if there is any opportunity for optimization.


## 5. Missing Data

Real datasets almost always have gaps. Pandas represents a missing value as `NaN` (or `NaT` for missing dates), and the first step is always the same: **find out where the gaps are before deciding what to do about them.**

- `.isnull()` / `.isna()` returns a same-shaped DataFrame of `True`/`False`; chaining `.sum()` turns that into a per-column count of missing values.
- Once you know *where* the nulls are, look at *why* — sometimes there's a pattern (e.g. all missing values belong to the same city) that suggests a smarter fix than a generic one.
- Two broad strategies: **drop** the rows/columns (`.dropna()`) when missing data is rare and unrecoverable, or **fill** them (`.fillna()`) when you can infer a sensible value. Dropping loses information; filling risks inserting a wrong assumption — the right call depends on context, not a fixed rule.

In this dataset, `Postal Code` has 11 missing values. Let's find them.

In [ ]:
# Will get the number of the empty cells with the missing values
df_raw.isnull().sum()

In [ ]:
# This will get the rows with the missing values
# Usuing this information in this case we can find the missing value
# Finally we can fill the missing places with this piece of information
df_raw[df_raw["Postal Code"].isnull() == True][["Country/Region", "City", "State", "Region"]]

In [ ]:
df_raw["Postal Code"].dtype

In [ ]:
# Now, we are going to work on the postal code column
# First, we need to convert it into a string data type, why?! because it does not make any sense to have it numeric
# Second, we need to fill the missing data through investigating these values
df_raw["Postal Code"] = df_raw["Postal Code"].astype("str")

# Finding the count of the null values in the Postal Code column
df_raw["Postal Code"].isnull().sum()

# Investigating these null values to find the corresponding one - maybe from the internet.
df_raw[df_raw["Postal Code"].isnull() == True][["Country/Region", "City", "State", "Region"]]

# Filling the missing values and showing the final result
df_raw["Postal Code"] = df_raw["Postal Code"].fillna("05401")
df_raw.info()

### 🧪 Exercise 5 — Missing Data

1. Re-run `.isnull().sum()` on the whole DataFrame — confirm every column now shows 0.
2. Suppose instead of filling `Postal Code`, we wanted to simply drop those rows. Write the `.dropna()` call that would do that (don't apply it — just write and run it as a *new* DataFrame so `df` stays intact).
3. In your own words (as a comment), when would dropping rows be the *wrong* choice even if it's the easiest one?

In [ ]:
# TODO 1: confirm no missing values remain


# TODO 2: write (but don't overwrite df with) a dropna() call for Postal Code


# TODO 3: comment — when is dropping rows the wrong call?


## 6. Duplicate Detection

Duplicate rows silently distort totals and averages, so it's worth explicitly checking for them rather than assuming a CSV is clean.

- `.duplicated()` flags rows that are **exact repeats** of an earlier row across *every* column; `.sum()` counts them.
- Exact duplicates are the easy case. More often, "duplicate" means something more specific to the business context — e.g. the same `Order ID` + `Product ID` combination appearing twice, which might indicate a data entry error even though the two rows aren't byte-for-byte identical (different Row ID, etc). Pass `subset=[...]` to check duplicates on specific columns only.
- Finding **zero** duplicates is a real, useful result — it doesn't mean the check was pointless, it means you've confirmed something instead of assumed it.

In [ ]:
# This is how to detect the duplicated rows >> fully duplicated rows...
df_raw.duplicated().sum()

In [ ]:
# from here we need to drop the fully duplicated rows
df_raw = df_raw.drop_duplicates()

In [ ]:
df_raw.shape

In [ ]:
# What if we need to detect the duplicated rows according to specific mix of columns
df_raw.duplicated(subset=["Order ID", "Product ID"]).sum()

In [ ]:
# So, the second stage is to explore these rows where the Order ID and Product ID repeat
# Did you get the point that if the order id and the product id are repeating then the customer is the same!
# So, we need to check the quantity of the sold product to be sure what's wrong with this mix
# I can replace the condition using the mask condition

# dup_mask = df_raw[df_raw.duplicated(subset=["Order ID", "Product ID"], keep=False) == True]
# order_product_duplicates = df_raw[dup_mask].sort_values("Order ID")

order_product_duplicates = df_raw[df_raw.duplicated(subset=["Order ID", "Product ID"], keep=False) == True].sort_values("Order ID")
order_product_duplicates

In [ ]:
# That said, I need to decide; are we going to keep the lowest or the highest >> keeping the highest will exagerate the sales
# Keeping the lower will drop the sales!!!!
# It seems that I am missing something big ...
# If you divided the sales / quantity > unit price
# If you divided the profit / quantity > unit_profit
# If the unit_price and the unit_profit are identical for duplicated rows
# then it will not be a mistakenly duplicted rows but instead they should
# be valid orders shipped to the same customer
order_product_duplicates["Unit Price"] = order_product_duplicates["Sales"] / order_product_duplicates["Quantity"]
order_product_duplicates["Unit Profit"] = order_product_duplicates["Profit"] / order_product_duplicates["Quantity"]

In [ ]:
# As you can see in the resulted DataFrame the unit_price and the unit_profit are identical ...
# So, these duplications are legitmate!
order_product_duplicates

### 🧪 Exercise 6 — Duplicate Detection

1. Check for duplicates using only the `Customer ID` and `Order Date` columns — how many rows come back as duplicates? (Think about what a "duplicate" even means here before assuming it's a problem.)
2. Using `keep="first"` vs `keep="last"` in `.duplicated()`, explain in a comment what the difference is.
3. If you did want to remove duplicate rows, which pandas method would you use (look it up if you're not sure — hint: it's a close cousin of `.duplicated()`)?

In [ ]:
# TODO 1: duplicated() on Customer ID + Order Date
df_raw.duplicated(subset=["Customer ID", "Order Date"], keep=False).sum()

In [ ]:
# TODO 2: comment explaining keep="first" vs keep="last"
# If we'd like to find the duplicated rows according to a specific pair then we use this subset option
# Keeping the first row as a default value or last or False to remove all rows
# implementing the change into the DataFrame directly which is not favorable
# Ignoring index = True means that the index will be reset in order not to have gaps
# df_raw.drop_duplicates(subset=, keep='first', inplace=, ignore_index=)

In [ ]:

# TODO 3: name (and try) the method that would actually remove duplicate rows


## 7. Inconsistent / Unclean Values

Beyond missing values and duplicate rows, text columns are prone to *inconsistency*: the same real-world value spelled, capitalized, or spaced differently across rows (`"Consumer"` vs `"consumer "` vs `"CONSUMER"`). Pandas will treat each of those as a *different* category, which quietly breaks `.value_counts()` and `.groupby()` results.

The habit to build: run `.unique()` on every categorical column and actually read the output, even when you expect it to be clean. Common fixes when something *is* off:

- `.str.strip()` — remove leading/trailing whitespace
- `.str.lower()` / `.str.title()` — normalize casing
- `.replace({...})` — map known typos/variants to a single canonical value

This dataset happens to be clean on this front — which is itself worth confirming rather than assuming.

In [ ]:
# Selecting all the 'category' columns
category_col = df_raw.select_dtypes('category').columns.tolist()

In [ ]:
# Check every key categorical column for stray variants (casing, whitespace, typos)
# Through checking the unique values we are able to define the misspelling...
for col in category_col:
    print(col, "->", df[col].unique())

In [ ]:
# Even on a clean column, this is the defensive pattern to reach for
# whenever you *do* find inconsistent casing/whitespace:
df["Segment"] = df["Segment"].astype(str).str.strip().str.title()
df["Segment"].unique()

> Regarding this step - data incosistency and unclean data values; we need to move it to the beginning of the notebook in order to mistakenly corrupt the data conversion step. So, you wil find it in the top of the notebook and for sanity check it will be here too.

### 🧪 Exercise 7 — Inconsistent / Dirty Values

1. Run `.unique()` on the `Sub-Category` column. Are there any surprises?
2. Simulate a dirty value: create a small test Series like `pd.Series([" consumer", "Consumer ", "CONSUMER"])` and clean it into a single consistent value using `.str.strip()` and `.str.title()` (or `.str.lower()`).
3. In your own words (as a comment): why can inconsistent text values silently break a `.groupby()` later on, even though the data "looks" fine at a glance?

In [ ]:
# TODO 1: unique() on Sub-Category


# TODO 2: clean the simulated dirty Series into one consistent value


# TODO 3: comment — why does inconsistent text break groupby later?


## 8. Outliers & Sanity Checks

Not every extreme value is an error — but every extreme value deserves a look. This is where domain knowledge matters as much as code:

- **Range checks**: `Discount` should logically fall between 0 and 1 (0%–100%) — confirming that with `.min()`/`.max()` is a one-line sanity check.
- **Negative values that are *expected***: negative `Profit` looks alarming at first, but in a sales dataset it's a completely legitimate result (an order sold at a loss) — not a data error. The lesson: an "outlier" is only a *mistake* if it violates a real-world constraint, not just because it's far from the mean.
- **Visual checks**: boxplots and histograms make outliers on continuous columns (like `Sales`) easy to spot visually, and the IQR (interquartile range) method gives a numeric way to flag them.

In [ ]:
# Range sanity check: Discount should never be negative or above 1 (100%)
print("Discount min/max:", df["Discount"].min(), df["Discount"].max())

# Negative profit is common in this dataset -- a real business signal, not an error
print("Orders sold at a loss:", (df["Profit"] < 0).sum())

**I think doing the distribution analysis using the plotbox analysis while the data contains the loss and profit at the same dataset does not make any sense. So, I need to separate both of them.**

In [ ]:
consumer_mask = ((df_raw["Segment"] == 'Consumer') & (df_raw["Profit"] >= 0))
corporate_mask = ((df_raw["Segment"] == 'Corporate') & (df_raw["Profit"] >= 0))
home_office_mask = ((df_raw["Segment"] == 'Home Office') & (df_raw["Profit"] >= 0))

consumer_sales = df_raw[consumer_mask]
corporate_sales = df_raw[corporate_mask]
home_office_sales = df_raw[home_office_mask]

print(consumer_sales.shape, corporate_sales.shape, home_office_sales.shape)

In [ ]:
home_office_sales["Profit"].min()

In [ ]:
# Visual check for outliers in Sales using a boxplot
plt.figure(figsize=(12, 4))
plt.boxplot(consumer_sales["Sales"], orientation="horizontal")
plt.title("Sales distribution -- boxplot")
plt.xlabel("Sales")
plt.show()

In [ ]:
# IQR method: flag numeric outliers beyond 1.5x the interquartile range
q1 = consumer_sales["Sales"].quantile(0.25)
q3 = consumer_sales["Sales"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = consumer_sales[(consumer_sales["Sales"] < lower_bound) | (consumer_sales["Sales"] > upper_bound)]
print(f"Sales outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print("Number of Sales outliers:", len(outliers))

In [ ]:
# Visual check for outliers in Sales using a boxplot
plt.figure(figsize=(12, 4))
plt.boxplot(corporate_sales["Sales"], orientation="horizontal")
plt.title("Sales distribution -- boxplot")
plt.xlabel("Sales")
plt.show()

In [ ]:
# IQR method: flag numeric outliers beyond 1.5x the interquartile range
q1 = corporate_sales["Sales"].quantile(0.25)
q3 = corporate_sales["Sales"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = corporate_sales[(corporate_sales["Sales"] < lower_bound) | (corporate_sales["Sales"] > upper_bound)]
print(f"Sales outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print("Number of Sales outliers:", len(outliers))

In [ ]:
# Visual check for outliers in Sales using a boxplot
plt.figure(figsize=(12, 4))
plt.boxplot(home_office_sales["Sales"], orientation="horizontal")
plt.title("Sales distribution -- boxplot")
plt.xlabel("Sales")
plt.show()

In [ ]:
# IQR method: flag numeric outliers beyond 1.5x the interquartile range
q1 = home_office_sales["Sales"].quantile(0.25)
q3 = home_office_sales["Sales"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = home_office_sales[(home_office_sales["Sales"] < lower_bound) | (home_office_sales["Sales"] > upper_bound)]
print(f"Sales outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print("Number of Sales outliers:", len(outliers))

### 🧪 Exercise 8 — Outliers & Sanity Checks

1. Apply the same IQR outlier method shown above, but to the `Profit` column instead of `Sales`.
2. Draw a boxplot for `Quantity` — does it show many outliers? What does that tell you about how order quantities are distributed?
3. Is a `Quantity` of 0 or negative possible in this dataset? Check with a range sanity check, similar to the `Discount` check above.

In [ ]:
# TODO 1: IQR outlier method applied to Profit


# TODO 2: boxplot for Quantity


# TODO 3: range sanity check on Quantity (min/max)


In [ ]:
# This is the final step we are going to conduct which is exporting the final version of the DataFrame
# into a new CSV file - remember to ignore the index storing - and this new CSV file should be
# The entry point for the next phase which is the feature engineering and the data analysis pahse
# We have an issue with exporting the data into CSV file; the csv format does not preserve the Pandas data types
# So, exporting to the CSV will get us back one step backward while we are working on the feature engineering
# and data analysis pahse. So, exporting to .parquet or .pickle file types will preserve the data types of Pandas
# But to do this we need first to install the pyarrow package and import it then use the to_parquet() function.

df_raw.to_parquet("Sample_Superstore_2019_Clean.parquet", index=False, engine='pyarrow')

## 9. Wrap-Up — A Repeatable Cleaning Checklist

Every dataset is different, but the *process* for approaching one is repeatable. Whenever you open a new CSV, work through the same sequence we just walked through:

1. **Shape & types** — `.shape`, `.info()`, `.dtypes`. What am I working with?
2. **Descriptive stats** — `.describe()`, `.value_counts()`. What does "normal" look like here?
3. **Fix types** — convert dates, numbers, and categories to the dtype they actually represent.
4. **Missing data** — `.isnull().sum()`, then decide drop vs. fill *per column*, based on why it's missing.
5. **Duplicates** — `.duplicated()`, both full-row and on meaningful subsets of columns.
6. **Consistency** — `.unique()` on every categorical column; normalize casing/whitespace/typos.
7. **Outliers** — range checks against domain knowledge, plus boxplots/IQR for a numeric view.
8. **Filter & select** — `.loc`/`.iloc`/boolean indexing to act on everything found above.

### 🧪 Capstone Exercise

Using everything from this notebook, answer the following in one code cell each:

1. Which `State` generated the highest total `Profit`, and which generated the largest total *loss*?
2. Is there a relationship between `Discount` and `Profit`? (Hint: group `Discount` into bins with `pd.cut()`, then look at average `Profit` per bin.)
3. Pick one thing you noticed while exploring this dataset that *wasn't* covered explicitly in this notebook, and investigate it your own way.

In [ ]:
# TODO 1: State with highest total Profit, and State with largest total loss


# TODO 2: relationship between Discount (binned) and average Profit


# TODO 3: your own exploration


> By ending of this notebook we should save the df_clean into a new CSV file where we can access it from the feature and data analysis notebook.